In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib


C:\Users\Jayesh Sanjay Patil\anaconda3\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
C:\Users\Jayesh Sanjay Patil\anaconda3\lib\site-packages\numpy\.libs\libopenblas.EL2C6PLE4ZYW3ECEVIV3OXXGRN2NRFM2.gfortran-win_amd64.dll
C:\Users\Jayesh Sanjay Patil\anaconda3\lib\site-packages\numpy\.libs\libopenblas64__v0.3.21-gcc_10_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [2]:
orders = pd.read_csv("../Data/Raw/olist_orders_dataset.csv")
customers = pd.read_csv("../Data/Raw/olist_customers_dataset.csv")
items = pd.read_csv("../Data/Raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../Data/Raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../Data/Raw/olist_order_reviews_dataset.csv")
products = pd.read_csv("../Data/Raw/olist_products_dataset.csv")
geolocation = pd.read_csv("../Data/Raw/olist_geolocation_dataset.csv")
sellers = pd.read_csv("../Data/Raw/olist_sellers_dataset.csv")
proc_cat = pd.read_csv("../Data/Raw/product_category_name_translation.csv")

In [3]:
payments_clean = payments.groupby("order_id", as_index=False).agg({
    "payment_value": "sum",
    "payment_installments": "max"
})

In [4]:
df = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(payments_clean, on="order_id", how="left")
)
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_value,payment_installments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,3.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62,1.0


In [5]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 99441 entries, 0 to 99440
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 8   customer_unique_id             99441 non-null  object        
 9   customer_zip_code_prefix       99441 non-null  int64         
 10  customer_city                  99441 non-null  object        
 11  customer_state 

In [7]:
df_items = (
    df
    .merge(items, on="order_id", how="left")
    .merge(products, on="product_id", how="left")
)

In [8]:
df.duplicated(subset=["order_id"]).sum()

0

### The data which was at payment-level which is converted in order level
- Many orders where the mode of payment was more than one where consider different orders, after classification on the order ID basis( it is always unique for orders), the multiple payments from different gateway are collected all together.

In [9]:
df_items["delivery_days"] = (
    df_items["order_delivered_customer_date"] - 
    df_items["order_purchase_timestamp"]
).dt.days

df_items["delay_days"] = (
    df_items["order_delivered_customer_date"] - 
    df_items["order_estimated_delivery_date"]
).dt.days

In [10]:
df_items["order_value"] = df_items.groupby("order_id")["price"].transform("sum")

In [11]:
df_items["total_items"] = df_items.groupby("order_id")["order_item_id"].transform("count")

In [12]:
df_items.shape

(113425, 32)

In [13]:
df_items.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 113425 entries, 0 to 113424
Data columns (total 32 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113425 non-null  object        
 1   customer_id                    113425 non-null  object        
 2   order_status                   113425 non-null  object        
 3   order_purchase_timestamp       113425 non-null  datetime64[ns]
 4   order_approved_at              113264 non-null  datetime64[ns]
 5   order_delivered_carrier_date   111457 non-null  datetime64[ns]
 6   order_delivered_customer_date  110196 non-null  datetime64[ns]
 7   order_estimated_delivery_date  113425 non-null  datetime64[ns]
 8   customer_unique_id             113425 non-null  object        
 9   customer_zip_code_prefix       113425 non-null  int64         
 10  customer_city                  113425 non-null  object        
 11  

In [14]:
df_items["price"].sum()

13591643.7

In [15]:
df_items["order_id"].nunique()

99441

In [16]:
df_items.groupby("order_id")["price"].sum().mean()

136.68048088828712

In [17]:
df_items["total_items"].mean()

1.386131805157593

In [18]:
df_items.groupby("product_category_name")["price"]\
    .sum()\
    .sort_values(ascending=False)\
    .head(10)

product_category_name
beleza_saude              1258681.34
relogios_presentes        1205005.68
cama_mesa_banho           1036988.68
esporte_lazer              988048.97
informatica_acessorios     911954.32
moveis_decoracao           729762.49
cool_stuff                 635290.85
utilidades_domesticas      632248.66
automotivo                 592720.11
ferramentas_jardim         485256.46
Name: price, dtype: float64

In [19]:
df_items["order_month"] = df_items["order_purchase_timestamp"].dt.to_period("M")

df_items.groupby("order_month")["price"].sum()

order_month
2016-09        267.36
2016-10      49507.66
2016-12         10.90
2017-01     120312.87
2017-02     247303.02
2017-03     374344.30
2017-04     359927.23
2017-05     506071.14
2017-06     433038.60
2017-07     498031.48
2017-08     573971.68
2017-09     624401.69
2017-10     664219.43
2017-11    1010271.37
2017-12     743914.17
2018-01     950030.36
2018-02     844178.71
2018-03     983213.44
2018-04     996647.75
2018-05     996517.68
2018-06     865124.31
2018-07     895507.22
2018-08     854686.33
2018-09        145.00
2018-10          0.00
Freq: M, Name: price, dtype: float64

In [20]:
df_items["delay_days"].describe()

count    110196.000000
mean        -12.030201
std          10.160157
min        -147.000000
25%         -17.000000
50%         -13.000000
75%          -7.000000
max         188.000000
Name: delay_days, dtype: float64

In [21]:
snapshot_date = df_items["order_purchase_timestamp"].max()
snapshot_date

Timestamp('2018-10-17 17:30:18')

## Recency Frequency Monetory
 - Recency - Days since Last Purchased 
 - Frequency - Number of times purchased
 - Monetory - Total money spent

In [22]:
rfm = df_items.groupby("customer_unique_id").agg({
    "order_purchase_timestamp": lambda x: (snapshot_date - x.max()).days,
    "order_id": "nunique",
    "price": "sum"
}).reset_index()

In [23]:
rfm.columns = ["customer_id", "recency", "frequency", "monetary"]

In [24]:
rfm.head()

,customer_id,recency,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,160,1,129.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,163,1,18.90
2,0000f46a3911fa3c0805444483337064,585,1,69.00
3,0000f6ccb0745a6a4b88665a16c9f078,369,1,25.99
4,0004aac84e0df4da2b147fca70cf8255,336,1,180.00


In [25]:
rfm.describe()

,recency,frequency,monetary
count,96096.000000,96096.000000,96096.000000
mean,287.735691,1.034809,141.438184
std,153.414676,0.214384,217.215904
min,0.000000,1.000000,0.000000
25%,163.000000,1.000000,45.990000
50%,268.000000,1.000000,89.000000
75%,397.000000,1.000000,154.000000
max,772.000000,17.000000,13440.000000


### Recency Score (LOW is GOOD)

In [26]:
rfm["R_score"] = pd.qcut(rfm["recency"], 5, labels=[5,4,3,2,1])

### Frequency Score (HIGH is GOOD)

In [27]:
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1,2,3,4,5])

### Monetary Score (HIGH is GOOD)

In [28]:
rfm["M_score"] = pd.qcut(rfm["monetary"], 5, labels=[1,2,3,4,5])

In [29]:
rfm["RFM_score"] = rfm["R_score"].astype(str) + rfm["F_score"].astype(str) + rfm["M_score"].astype(str)

In [30]:
def segment_customer(row):
    if row["R_score"] >= 4 and row["F_score"] >= 4:
        return "Champions"
    elif row["R_score"] >= 3 and row["F_score"] >= 3:
        return "Loyal Customers"
    elif row["R_score"] >= 4:
        return "Recent Customers"
    elif row["F_score"] >= 4:
        return "Frequent Customers"
    elif row["M_score"] >= 4:
        return "Big Spenders"
    else:
        return "At Risk"

rfm["segment"] = rfm.apply(segment_customer, axis=1)

In [31]:
rfm["segment"].value_counts()

Loyal Customers       19237
At Risk               19092
Champions             15453
Recent Customers      15449
Frequent Customers    15285
Big Spenders          11580
Name: segment, dtype: int64

In [32]:
rfm_revenue = rfm.merge(
    df_items.groupby("customer_unique_id")["price"].sum().reset_index(),
    left_on="customer_id",
    right_on="customer_unique_id",
    how="left"
)

rfm_revenue.groupby("segment")["price"].sum().sort_values(ascending=False)

segment
Big Spenders          3173028.65
Loyal Customers       2641917.60
Champions             2343105.55
Frequent Customers    2222772.04
Recent Customers      2161606.69
At Risk               1049213.17
Name: price, dtype: float64

In [33]:
df_items = df_items.merge(reviews, on="order_id", how="left")

In [34]:
df_items["review_score"].value_counts()

5.0    63596
4.0    21348
1.0    14775
3.0     9476
2.0     3936
Name: review_score, dtype: int64

In [35]:
df_items.groupby("delay_days")["review_score"].mean()

delay_days
-147.0    5.0
-140.0    5.0
-135.0    3.0
-124.0    5.0
-109.0    5.0
         ... 
 166.0    1.0
 167.0    1.0
 175.0    1.0
 181.0    NaN
 188.0    2.0
Name: review_score, Length: 198, dtype: float64

In [36]:
df_items.groupby("product_category_name")["review_score"].mean().sort_values()

product_category_name
seguros_e_servicos                               2.500000
fraldas_higiene                                  3.256410
portateis_cozinha_e_preparadores_de_alimentos    3.266667
pc_gamer                                         3.333333
moveis_escritorio                                3.493183
                                                   ...   
flores                                           4.419355
construcao_ferramentas_ferramentas               4.444444
livros_interesse_geral                           4.446266
fashion_roupa_infanto_juvenil                    4.500000
cds_dvds_musicais                                4.642857
Name: review_score, Length: 73, dtype: float64

#### 

In [37]:
df_items.to_csv("../Data/Processed/ecommerce_master.csv", index=False)

In [38]:
rfm.to_csv("../Data/Processed/rfm_data.csv", index=False)

In [39]:
df_ml = df_items.copy()

df_ml["satisfaction_label"] = df_ml["review_score"].apply(
    lambda x: 1 if x >= 4 else 0
)

In [40]:
features = [
    "order_value",
    "total_items",
    "freight_value",
    "payment_installments",
    "delivery_days",
    "delay_days",
    "product_category_name",
    "customer_state",
    "product_weight_g"
]

In [41]:
df_model = df_ml[features + ["satisfaction_label"]]

In [42]:
df_model.isnull().sum()

order_value                 0
total_items                 0
freight_value             778
payment_installments        3
delivery_days            3253
delay_days               3253
product_category_name    2390
customer_state              0
product_weight_g          796
satisfaction_label          0
dtype: int64

In [43]:
num_cols = [
    "order_value",
    "total_items",
    "freight_value",
    "payment_installments",
    "delivery_days",
    "delay_days",
    "product_weight_g"
]

for col in num_cols:
    df_model[col] = df_model[col].fillna(df_model[col].median())

C:\Users\Jayesh Sanjay Patil\AppData\Local\Temp\ipykernel_19820\3003202811.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model[col] = df_model[col].fillna(df_model[col].median())


In [44]:
cat_cols = [
    "product_category_name",
    "customer_state"
]

for col in cat_cols:
    df_model[col] = df_model[col].fillna("Unknown")

C:\Users\Jayesh Sanjay Patil\AppData\Local\Temp\ipykernel_19820\3336198175.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model[col] = df_model[col].fillna("Unknown")


In [45]:
df_model["satisfaction_label"].value_counts(normalize=True)

1    0.744522
0    0.255478
Name: satisfaction_label, dtype: float64

In [46]:
df_model = pd.get_dummies(
    df_model,
    columns=["product_category_name", "customer_state"],
    drop_first=True
)

In [47]:
X = df_model.drop("satisfaction_label", axis=1)
y = df_model["satisfaction_label"]

In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [49]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [50]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance.head(15))

                                          Feature  Importance
5                                      delay_days    0.164630
4                                   delivery_days    0.146205
0                                     order_value    0.139754
2                                   freight_value    0.130028
6                                product_weight_g    0.117954
3                            payment_installments    0.057321
1                                     total_items    0.048725
104                             customer_state_SP    0.008136
89                              customer_state_MG    0.008086
79    product_category_name_utilidades_domesticas    0.006826
61         product_category_name_moveis_decoracao    0.006358
51   product_category_name_informatica_acessorios    0.006312
97                              customer_state_RJ    0.006162
39            product_category_name_esporte_lazer    0.005951
20          product_category_name_cama_mesa_banho    0.005788


In [51]:
y_pred = model.predict(X_test)

In [52]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.47      0.60      5830
           1       0.84      0.97      0.90     16989

    accuracy                           0.84     22819
   macro avg       0.84      0.72      0.75     22819
weighted avg       0.84      0.84      0.82     22819



In [53]:
joblib.dump(model, "../Models/customer_satisfaction_model.pkl")
joblib.dump(X.columns.tolist(), "../Models/model_features.pkl")

['../Models/model_features.pkl']